# 01a — Databricks Ray Dry Run (no Flux)

**Goal:** validate the cluster + Ray plumbing in ~5 min for ~$1 of cluster time, *before* committing to the full ~20 min Flux smoke (`01_databricks_ray_smoke.ipynb`).

**What this verifies:**
1. Repo installs cleanly (`pip install -e ..`)
2. `cap`, `torch`, `ray` import
3. All 4 L4 GPUs are visible to the cluster
4. `ray.util.spark.setup_ray_cluster` brings up Ray and reports `GPU: 4.0` in cluster resources
5. Ray cluster shuts down cleanly

**What this does NOT do:**
- Download Flux weights (~24 GB)
- Load any model into VRAM
- Generate any image

If every cell runs green → run `01_databricks_ray_smoke.ipynb` for the real smoke. If any cell fails → fix that before paying for Flux.

## 1. Install the repo

In [ ]:
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, "-m", "pip", "show", "cap"], capture_output=True, text=True)
print("pip show cap rc:", res.returncode)
print(res.stdout if res.stdout else "(no stdout)")
print(res.stderr if res.stderr else "(no stderr)")
print("---")
print("python:", sys.executable)
print("sys.path[:5]:", sys.path[:5])

## 2. Imports + GPU sanity

In [ ]:
import sys
import torch
import ray

import cap
from cap.generator import FluxPuLIDControlNetGenerator, GenerationRequest  # import-only check, no instantiation

print(f"python:    {sys.version.split()[0]}")
print(f"cap:       {cap.__version__}")
print(f"torch:     {torch.__version__}")
print(f"cuda:      {torch.version.cuda}")
print(f"ray:       {ray.__version__}")
print(f"GPU count on driver: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"Driver GPU: {p.name} | {p.total_memory / 1e9:.1f} GB VRAM")

## 3. Bring up Ray across all 4 L4s

`num_gpus_head_node=1` pulls the driver L4 into the actor pool. Combined with 3 worker L4s → `GPU: 4.0` in `ray.cluster_resources()`.

In [ ]:
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

setup_ray_cluster(
    num_worker_nodes=3,
    num_gpus_worker_node=1,
    num_cpus_worker_node=8,
    num_gpus_head_node=1,
    num_cpus_head_node=4,
)

ray.init(ignore_reinit_error=True)
print("Ray cluster resources:")
for k, v in sorted(ray.cluster_resources().items()):
    print(f"  {k}: {v}")

assert ray.cluster_resources().get("GPU", 0) >= 3, "Expected at least 3 GPUs visible to Ray"
print("\nGPU count check: OK")

## 4. Probe each GPU from a Ray task

Schedule one trivial task per GPU (just reads `nvidia-smi` from inside the task). Confirms each L4 is reachable from a Ray actor / task and reports its name + free memory.

In [ ]:
@ray.remote(num_gpus=1)
def probe_gpu():
    import socket, torch
    p = torch.cuda.get_device_properties(0)
    return {
        "host": socket.gethostname(),
        "gpu": p.name,
        "vram_gb": round(p.total_memory / 1e9, 1),
        "cuda_visible": torch.cuda.is_available(),
    }

n_gpus = int(ray.cluster_resources().get("GPU", 0))
probes = ray.get([probe_gpu.remote() for _ in range(n_gpus)])
for i, p in enumerate(probes):
    print(f"  actor {i}: {p}")

## 5. Cleanup

In [ ]:
shutdown_ray_cluster()
print("Ray cluster shut down. Dry run complete.")